# Tutorial -  Retornos
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera
import statsmodels.api as sm
from scipy import stats

## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde Yahoo Finance

https://finance.yahoo.com/


In [ ]:
# Descargamos datos de la acción sleccionada:
df = yfin.download('^GSPC', start='2021-01-01', multi_level_index=False)
df

## 3. Visualización y Estadísticas Descriptivas

### 3.1. Utiliza la librería mpf para hacer un gráfico de velas japonesas de la acción

In [ ]:
mpf.plot(df,type='candle', volume=True,figratio=(19,8),style='yahoo',title='S&P 500')

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df["Log Returns"])*252
LogReturns

In [ ]:
LogReturns*50

In [ ]:
1000*np.exp(LogReturns*50)

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'red')
plt.title('Retornos Logarítmicos de S&P 500')
plt.ylabel('Retornos')
plt.xlabel('Fecha')
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:  

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log Returns'])

# Anualizamos la volatilidad diaria.
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*vol_a))

## 6. Normalidad de los retornos

### 6.1. Histogram

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(df['Log Returns'])))

plt.hist(df['Log Returns'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Log-Returns')

# Show the plot
plt.show()

### 6.2. Jarque-Bera test

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = stats.jarque_bera(df['Log Returns'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

## 7. Autocorrelación

### 7.1. Autocorrelation

In [ ]:
# Crear el autocorrelograma
plot_acf(df['Log Returns'], lags = 21)
plt.show()

In [ ]:
df['Log Returns^2'] = df['Log Returns']**2

In [ ]:
# Crear el autocorrelograma de retornos al cuadrado
plot_acf(df['Log Returns^2'], lags = 21)
plt.show()

### 7.2. Partial Autocorrelation

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns'], lags = 21)
plt.show()

### 7.3. Ljung-Box test

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 5
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 5
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns^2'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")